In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {df_food.shape}")
df_food.head()

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
# delivery_time distribution (target variable)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_food = df_food.drop('Order_ID' , axis=1)
df_food


In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (df_food.isnull().sum() / len(df_food)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)


In [ ]:

df_food['Delivery_Time']= df_food['Delivery_Time'].fillna(df_food['Delivery_Time'].mean())
df_food['Weather']= df_food['Weather'].fillna(df_food['Weather'].mode()[0])
df_food['Traffic_Level']= df_food['Traffic_Level'].fillna(df_food['Traffic_Level'].mode()[0])
df_food['Time_of_Day']= df_food['Time_of_Day'].fillna(df_food['Time_of_Day'].mode()[0])
df_food['Courier_Experience_yrs']= df_food['Courier_Experience_yrs'].fillna(df_food['Courier_Experience_yrs'].mean())

print("Missing values remaining:", df_food.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df_food):
  duplicates = df_food.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_food.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_food)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder #import LabelEncoder

categorical_cols = df_food.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    # TODO: Apply fit_transform to encode the column
    df_food[col] = le.fit_transform(df_food[col])

df_food.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

numerical_cols =  df_food.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

# TODO: Apply fit_transform to scale the numerical columns
df_food[numerical_cols] = scaler.fit_transform(df_food[numerical_cols])

df_food.head()

In [ ]:
# Task 6: Write your code here:
# 1. Is the target imbalanced?
import seaborn as sns
def check_target_imbalance(df_food, target_column):
    print("TargetDistribution:")
    print(df_food[target_column].value_counts(normalize=True))
    sns.countplot(x=df_food[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(df_food, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split
X = df_food.drop("Delivery_Time",axis=1)
y = df_food['Delivery_Time']
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,shuffle=True)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)

mae_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X, y), 1):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)
    print(f"Fold {fold}: MAE = {mae:.4f}")

print(f"\nAverage MAE: {np.mean(mae_scores):.4f}")

In [ ]:
# Task 1: Write your code here:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importances['feature'][:15], importances['importance'][:15])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 15 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
all_predictions = model.predict(X)

plt.figure(figsize=(10, 5))
plt.hist(all_predictions, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Predicted Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.title('Distribution of Predicted Delivery Times')
plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here:
!pip install catboost
from catboost import CatBoostRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
cb_model = CatBoostRegressor(iterations=100, random_state=42, verbose=0)

ensemble_mae_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X, y), 1):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    rf_model.fit(X_train, y_train)
    cb_model.fit(X_train, y_train)

    rf_pred = rf_model.predict(X_test)
    cb_pred = cb_model.predict(X_test)

    avg_pred = (rf_pred + cb_pred) / 2

    mae = mean_absolute_error(y_test, avg_pred)
    ensemble_mae_scores.append(mae)
    print(f"Fold {fold}: Ensemble MAE = {mae:.4f}")

print(f"\nAverage Ensemble MAE: {np.mean(ensemble_mae_scores):.4f}")
print(f"Single Model MAE: {np.mean(mae_scores):.4f}")
print(f"Improvement: {(np.mean(mae_scores) - np.mean(ensemble_mae_scores)):.4f}")